# ReliPose-HOI Colab Launcher

This notebook launches and validates the compact research implementation. It does not define model classes or duplicate repository code. Long training cells are guarded by `RUN_LONG_TRAINING = False`.

## 1. Repository Clone/Open

In [ ]:
from pathlib import Path
REPO_URL = ""  # optional: set to your Git URL
REPO_DIR = Path('/content/ReliPose-HOI')
if not REPO_DIR.exists():
    if not REPO_URL:
        raise RuntimeError('Set REPO_URL or upload/open the repository at /content/ReliPose-HOI')
    !git clone {REPO_URL} {REPO_DIR}
%cd /content/ReliPose-HOI

## 2. Package Installation

In [ ]:
!pip install -e .[train]

## 3. Torch, Torchvision, CUDA/GPU Check

In [ ]:
import sys, torch
print('Python:', sys.version)
print('torch:', torch.__version__)
try:
    import torchvision
    print('torchvision:', torchvision.__version__)
except Exception as exc:
    raise RuntimeError('torchvision is required for real detected-box Colab runs; install a version compatible with the existing Colab torch package.') from exc
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Editable Paths And Run Flags

In [ ]:
RUN_LONG_TRAINING = False
RUN_TINY_OVERFIT = False
COCO_IMAGE_ROOT = '/content/drive/MyDrive/datasets/coco/train2017'
COCO_ANNOTATION_FILE = '/content/drive/MyDrive/datasets/coco/annotations/person_keypoints_train2017.json'
HICO_TRAIN_IMAGE_ROOT = '/content/drive/MyDrive/datasets/hico/images/train2015'
HICO_TRAIN_ANNOTATION_FILE = '/content/drive/MyDrive/datasets/hico/annotations/train.json'
OUTPUT_DIR = '/content/drive/MyDrive/relipose_outputs'
CHECKPOINT_PATH = f'{OUTPUT_DIR}/latest.pt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 6. Smoke Test Command

In [ ]:
!python scripts/smoke_test.py --config configs/smoke.yaml

## 7. COCO data check command and HICO data check command

In [ ]:
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path('configs/pose.yaml').read_text())
cfg['data']['coco_image_root'] = COCO_IMAGE_ROOT
cfg['data']['coco_annotation_file'] = COCO_ANNOTATION_FILE
Path('/content/relipose_pose_colab.yaml').write_text(yaml.safe_dump(cfg))
cfg = yaml.safe_load(Path('configs/hoi_oracle.yaml').read_text())
cfg['data']['hico_train_image_root'] = HICO_TRAIN_IMAGE_ROOT
cfg['data']['hico_train_annotation_file'] = HICO_TRAIN_ANNOTATION_FILE
Path('/content/relipose_hico_colab.yaml').write_text(yaml.safe_dump(cfg))
!python scripts/check_data.py --config /content/relipose_pose_colab.yaml --dataset coco
!python scripts/check_data.py --config /content/relipose_hico_colab.yaml --dataset hico

## 8. One-Batch Pose Forward/Backward

In [ ]:
!python scripts/train.py --config /content/relipose_pose_colab.yaml --stage pose --device {DEVICE} --max-samples 1 --max-steps 1 --overfit

## 9. One-Batch Oracle HOI Forward/Backward

In [ ]:
!python scripts/train.py --config /content/relipose_hico_colab.yaml --stage hoi_oracle --device {DEVICE} --max-samples 1 --max-steps 1 --overfit

## 10. Tiny Pose Overfit Command And Tiny Oracle HOI Overfit Command

In [ ]:
if RUN_TINY_OVERFIT:
    !python scripts/train.py --config /content/relipose_pose_colab.yaml --stage pose --device {DEVICE} --max-samples 2 --max-steps 20 --overfit
    !python scripts/train.py --config /content/relipose_hico_colab.yaml --stage hoi_oracle --device {DEVICE} --max-samples 2 --max-steps 20 --overfit

## 11. Checkpoint/Resume Test

In [ ]:
!python scripts/train.py --config /content/relipose_hico_colab.yaml --stage hoi_oracle --device {DEVICE} --max-samples 1 --max-steps 1 --overfit
!python scripts/train.py --config /content/relipose_hico_colab.yaml --stage hoi_oracle --device {DEVICE} --max-samples 1 --max-steps 1 --overfit --resume outputs/hoi_oracle/latest.pt

## 12. Full Training Commands Guarded By RUN_LONG_TRAINING

In [ ]:
if RUN_LONG_TRAINING:
    !python scripts/train.py --config /content/relipose_pose_colab.yaml --stage pose --device {DEVICE}
    !python scripts/train.py --config /content/relipose_hico_colab.yaml --stage hoi_oracle --device {DEVICE}
    !python scripts/train.py --config /content/relipose_hico_colab.yaml --stage robust --device {DEVICE} --resume outputs/hoi_oracle/latest.pt